##  Import and load

In [1]:
from datasets import load_from_disk
import pandas as pd
import numpy as np

DATASET_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k"

train = load_from_disk(f"{DATASET_PATH}/train")
validation = load_from_disk(f"{DATASET_PATH}/validation")
test = load_from_disk(f"{DATASET_PATH}/test")

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 45000
Validation: 2500
Test: 2500


In [2]:
print(train)
print(train.column_names)

Dataset({
    features: ['article', 'abstract'],
    num_rows: 45000
})
['article', 'abstract']


## Example of one sample

In [3]:
sample = train[0]

print("ARTICLE:")
print(sample["article"][:3000])

print("\n" + "="*80 + "\n")

print("ABSTRACT:")
print(sample["abstract"])

ARTICLE:
arp  220 is the nearest ( @xmath3 77  mpc ) example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels . 
 it contains two nuclei separated by 350  pc , both surrounded by massive discs of dense molecular gas ( e.g. , * ? ? ? 
 * ; * ? ? ? 
 * ; * ? ? ? 
 * ; * ? ? ? 
 * ; * ? ? ? 
 radio detections of supernovae at a rate of 13 yr@xmath4 @xcite confirm that huge populations of massive stars are present with an implied star formation rate ( sfr ) of @xmath5  yr@xmath4 . 
 although arp 220 could contain active galactic nuclei ( agns ) , particularly in the western nucleus , the observed supernova rates indicate that star formation provides a substantial fraction of the power radiated by the nuclei . 
 the nuclei of arp  220 provide access to the high - intensity mode of star formation in dense molecular media that appears to have been more common in young galaxies . 
 these types of environments are of special interest from a range of p

## Checking for missing data

In [4]:
def check_missing(dataset, name):
    empty_articles = 0
    empty_abstracts = 0

    for row in dataset:
        if not row["article"] or not row["article"].strip():
            empty_articles += 1

        if not row["abstract"] or not row["abstract"].strip():
            empty_abstracts += 1

    print(f"{name}:")
    print("  Empty articles:", empty_articles)
    print("  Empty abstracts:", empty_abstracts)

check_missing(train, "Train")
check_missing(validation, "Validation")
check_missing(test, "Test")

Train:
  Empty articles: 29
  Empty abstracts: 0
Validation:
  Empty articles: 0
  Empty abstracts: 0
Test:
  Empty articles: 0
  Empty abstracts: 0


## Calculating lengths

In [5]:
def length_stats(dataset, name):
    article_lengths = np.array([
        len(row["article"]) for row in dataset
    ])

    abstract_lengths = np.array([
        len(row["abstract"]) for row in dataset
    ])

    print(f"\n{name}")
    print("-" * 40)

    print("ARTICLE")
    print("Mean:", article_lengths.mean())
    print("Median:", np.median(article_lengths))
    print("Min:", article_lengths.min())
    print("Max:", article_lengths.max())

    print("\nABSTRACT")
    print("Mean:", abstract_lengths.mean())
    print("Median:", np.median(abstract_lengths))
    print("Min:", abstract_lengths.min())
    print("Max:", abstract_lengths.max())

length_stats(train, "TRAIN")


TRAIN
----------------------------------------
ARTICLE
Mean: 33913.11111111111
Median: 27668.5
Min: 0
Max: 502944

ABSTRACT
Mean: 1611.2502666666667
Median: 980.0
Min: 6
Max: 92698


In [6]:
articles = [row["article"] for row in train]

print("Total articles:", len(articles))
print("Unique articles:", len(set(articles)))
print("Duplicate articles:", len(articles) - len(set(articles)))

Total articles: 45000
Unique articles: 44971
Duplicate articles: 29


## Checking word counts

In [7]:
def word_count(text):
    return len(text.split())

article_word_counts = np.array([
    word_count(row["article"]) for row in train
])

abstract_word_counts = np.array([
    word_count(row["abstract"]) for row in train
])

print("ARTICLE WORD COUNTS")
print("Mean:", article_word_counts.mean())
print("Median:", np.median(article_word_counts))
print("Min:", article_word_counts.min())
print("Max:", article_word_counts.max())

print("\nABSTRACT WORD COUNTS")
print("Mean:", abstract_word_counts.mean())
print("Median:", np.median(abstract_word_counts))
print("Min:", abstract_word_counts.min())
print("Max:", abstract_word_counts.max())

ARTICLE WORD COUNTS
Mean: 6049.221
Median: 4925.0
Min: 0
Max: 97893

ABSTRACT WORD COUNTS
Mean: 278.4384888888889
Median: 164.0
Min: 2
Max: 18516


In [8]:
longest_indices = np.argsort(article_word_counts)[-10:][::-1]

for i in longest_indices:
    print(
        f"Index: {i}, "
        f"Words: {article_word_counts[i]:,}, "
        f"Characters: {len(train[i]['article']):,}"
    )

Index: 17946, Words: 97,893, Characters: 455,189
Index: 16567, Words: 88,872, Characters: 502,944
Index: 19344, Words: 87,653, Characters: 327,833
Index: 13107, Words: 65,716, Characters: 383,594
Index: 5072, Words: 65,275, Characters: 267,919
Index: 35956, Words: 64,749, Characters: 326,140
Index: 39517, Words: 63,529, Characters: 372,549
Index: 8333, Words: 63,363, Characters: 294,800
Index: 13822, Words: 62,911, Characters: 324,201
Index: 9839, Words: 58,616, Characters: 290,589


In [9]:
thresholds = [5000, 10000, 20000, 30000, 50000]

for threshold in thresholds:
    count = np.sum(article_word_counts > threshold)
    print(f"> {threshold:,} words: {count:,} documents")

> 5,000 words: 22,092 documents
> 10,000 words: 6,141 documents
> 20,000 words: 728 documents
> 30,000 words: 159 documents
> 50,000 words: 21 documents


## CLEANING THE DATASET

In [16]:
import re

def clean_text(text):
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"@xmath\d+", " [MATH] ", text)
    text = re.sub(r"@xcite", "", text)
    text = re.sub(r"\*\s*\?\s*\?\s*\?", "", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    text = re.sub(r"([.,;:!?])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [17]:
cleaned = clean_text(train[0]["article"])
print(cleaned[:3000])

arp 220 is the nearest ( [MATH] 77 mpc ) example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas ( e.g., *; *; *; *; radio detections of supernovae at a rate of 13 yr [MATH] confirm that huge populations of massive stars are present with an implied star formation rate ( sfr ) of [MATH] yr [MATH]. although arp 220 could contain active galactic nuclei ( agns ), particularly in the western nucleus, the observed supernova rates indicate that star formation provides a substantial fraction of the power radiated by the nuclei. the nuclei of arp 220 provide access to the high - intensity mode of star formation in dense molecular media that appears to have been more common in young galaxies. these types of environments are of special interest from a range of perspectives, including the information they can provide regarding the role of galactic winds,

In [18]:
original_lengths = []
cleaned_lengths = []

for row in train:
    original = row["article"]
    cleaned = clean_text(original)

    original_lengths.append(len(original))
    cleaned_lengths.append(len(cleaned))

original_lengths = np.array(original_lengths)
cleaned_lengths = np.array(cleaned_lengths)

removed = original_lengths - cleaned_lengths

print("Average characters removed:", removed.mean())
print("Median characters removed:", np.median(removed))
print("Maximum characters removed:", removed.max())
print("Average percentage removed:", (removed / original_lengths.clip(min=1) * 100).mean())

Average characters removed: 2063.5222222222224
Median characters removed: 1569.0
Maximum characters removed: 80860
Average percentage removed: 5.9180646725642


In [13]:
original_lengths = []
cleaned_lengths = []

for row in train:
    original = row["article"]
    cleaned = clean_text(original)

    original_lengths.append(len(original))
    cleaned_lengths.append(len(cleaned))

original_lengths = np.array(original_lengths)
cleaned_lengths = np.array(cleaned_lengths)

removed = original_lengths - cleaned_lengths

print("Average characters removed:", removed.mean())
print("Median characters removed:", np.median(removed))
print("Maximum characters removed:", removed.max())

Average characters removed: 4195.956444444444
Median characters removed: 2994.5
Maximum characters removed: 114744


In [19]:
from datasets import Dataset, DatasetDict

def clean_dataset(dataset):
    articles = []
    abstracts = []

    for row in dataset:
        article = clean_text(row["article"])
        abstract = clean_text(row["abstract"])

        if article and abstract:
            articles.append(article)
            abstracts.append(abstract)

    return {
        "article": articles,
        "abstract": abstracts
    }

In [20]:
cleaned_train = clean_dataset(train)
cleaned_validation = clean_dataset(validation)
cleaned_test = clean_dataset(test)

In [21]:
cleaned_dataset = DatasetDict({
    "train": Dataset.from_dict(cleaned_train),
    "validation": Dataset.from_dict(cleaned_validation),
    "test": Dataset.from_dict(cleaned_test)
})

print(cleaned_dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 44971
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 2500
    })
})


In [22]:
CLEANED_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k_cleaned"

cleaned_dataset.save_to_disk(CLEANED_PATH)

print("Saved successfully.")

Saving the dataset (0/4 shards):   0%|          | 0/44971 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

Saved successfully.


In [1]:
from datasets import load_from_disk
import numpy as np

CLEANED_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k_cleaned"

cleaned_dataset = load_from_disk(CLEANED_PATH)

train = cleaned_dataset["train"]
validation = cleaned_dataset["validation"]
test = cleaned_dataset["test"]

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 44971
Validation: 2500
Test: 2500


In [2]:
print(train[0]["article"][:3000])

print("\n" + "="*80 + "\n")

print(train[0]["abstract"])

arp 220 is the nearest ( [MATH] 77 mpc ) example of an ultraluminous infrared galaxy ( ulirg ) that supports star formation at extreme levels. it contains two nuclei separated by 350 pc, both surrounded by massive discs of dense molecular gas ( e.g., *; *; *; *; radio detections of supernovae at a rate of 13 yr [MATH] confirm that huge populations of massive stars are present with an implied star formation rate ( sfr ) of [MATH] yr [MATH]. although arp 220 could contain active galactic nuclei ( agns ), particularly in the western nucleus, the observed supernova rates indicate that star formation provides a substantial fraction of the power radiated by the nuclei. the nuclei of arp 220 provide access to the high - intensity mode of star formation in dense molecular media that appears to have been more common in young galaxies. these types of environments are of special interest from a range of perspectives, including the information they can provide regarding the role of galactic winds,

In [3]:
article_lengths = np.array([
    len(row["article"].split()) for row in train
])

abstract_lengths = np.array([
    len(row["abstract"].split()) for row in train
])

print("Article median:", np.median(article_lengths))
print("Article mean:", np.mean(article_lengths))
print("Article max:", np.max(article_lengths))

print("Abstract median:", np.median(abstract_lengths))
print("Abstract mean:", np.mean(abstract_lengths))
print("Abstract max:", np.max(abstract_lengths))

Article median: 4456.0
Article mean: 5489.3756198439
Article max: 88106
Abstract median: 153.0
Abstract mean: 254.87856618709836
Abstract max: 16890


In [4]:
for threshold in [5000, 10000, 20000, 30000, 50000]:
    count = np.sum(article_lengths > threshold)
    print(f"> {threshold:,} words: {count:,}")

> 5,000 words: 19,418
> 10,000 words: 4,721
> 20,000 words: 523
> 30,000 words: 117
> 50,000 words: 14
